In [ ]:
!pip install -q --upgrade bitsandbytes accelerate transformers==4.45.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 45.3 MB/s eta 0:00:00


In [ ]:
!pip install fpdf

  Preparing metadata (setup.py) ... done
  Created wheel for fpdf: filename=fpdf-1.7.2-py2.py3-none-any.whl size=40704 sha256=44f8376a4fcf7b97a9437890adc602300336a563d516c425e3f4a5696eb20bba
  Stored in directory: /root/.cache/pip/wheels/6e/62/11/dc73d78e40a218ad52e7451f30166e94491be013a7850b5d75
Successfully built fpdf


In [ ]:
import gradio as gr
import torch
import os
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import login
from fpdf import FPDF

In [ ]:
import torch
from google.colab import userdata
from huggingface_hub import login
import gradio as gr
import os

print("=== COLAB GPU DIAGNOSTICS ===")
!nvidia-smi
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM Free: {torch.cuda.memory_reserved(0)/1024**3:.1f} GB used")
else:
    print("❌ NO GPU! Restart runtime and select T4 again.")

=== COLAB GPU DIAGNOSTICS ===
Tue Apr 28 12:55:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------

In [ ]:
# ========================== 2. HF LOGIN ==========================
try:
    hf_token = userdata.get('HF_TOKEN')
    login(hf_token, add_to_git_credential=True)
    print("✅ HF login successful")
except:
    print("⚠️ No HF_TOKEN found in secrets. Make sure you added it.")

✅ HF login successful


In [ ]:
# ====================== LOAD MODELS ======================
print("🚀 Loading Whisper-medium.en...")
whisper_pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-medium.en",
    torch_dtype=torch.float16,
    device="cuda",
    return_timestamps=True
)

LLAMA = "meta-llama/Llama-3.2-3B-Instruct"
print("🚀 Loading Llama-3.2-3B (4-bit)...")
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    LLAMA, device_map="cuda", quantization_config=quant_config
)

print(f"✅ Llama loaded on: {model.device}")
print("🎉 All models ready!")


🚀 Loading Whisper-medium.en...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.06G [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/805 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [ ]:
# ====================== HELPER FUNCTIONS ======================
def clean_output(text):
    text = text.strip()
    prefixes = ["assistant", "Assistant:", "assistant:"]
    for p in prefixes:
        if text.lower().startswith(p.lower()):
            text = text[len(p):].strip()
    return text

def create_pdf(minutes_text):
    pdf = FPDF()
    pdf.add_page()
    pdf.set_margins(15, 15, 15)
    pdf.set_font("Arial", "B", 18)
    pdf.cell(0, 15, "Meeting Minutes", ln=True, align="C")
    pdf.ln(10)
    pdf.set_font("Arial", size=11)
    for line in minutes_text.split('\n'):
        line = line.strip()
        if not line:
            pdf.ln(6)
            continue
        if line.startswith("**") and line.endswith("**"):
            pdf.set_font("Arial", "B", 14)
            pdf.multi_cell(0, 8, line.replace("**", ""))
            pdf.set_font("Arial", size=11)
        else:
            pdf.multi_cell(0, 7, line)
    pdf.output("meeting_minutes.pdf")
    return "meeting_minutes.pdf"

# ====================== CORE FUNCTIONS ======================
def transcribe_audio(audio_path):
    return whisper_pipe(audio_path)["text"]

def generate_meeting_minutes(transcription):
    user_prompt = f"""Create clean professional meeting minutes in markdown format with exactly these sections:

    **Summary**
    **Key Discussion Points**
    **Main Takeaways**
    **Action Items** (with owners)

    Transcription:
    {transcription}"""

    messages = [
        {"role": "system", "content": "You are an expert assistant that writes clear, concise meeting minutes."},
        {"role": "user", "content": user_prompt}
    ]

    inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")

    outputs = model.generate(
        inputs,
        max_new_tokens=1200,
        do_sample=False,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_tokens = outputs[0][inputs.shape[1]:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return clean_output(response)

def process_meeting(audio, progress=gr.Progress()):
    if audio is None:
        return None, "❌ Upload audio", "**Error**", None

    progress(0.3, desc="🎙️ Transcribing...")
    transcription = transcribe_audio(audio)

    progress(0.6, desc="✍️ Generating minutes...")
    minutes = generate_meeting_minutes(transcription)

    progress(0.95, desc="📄 Creating PDF...")
    pdf_path = create_pdf(minutes)

    progress(1.0, desc="✅ Done!")
    return transcription, "✅ Complete!", minutes, pdf_path


In [ ]:
# ====================== UI ======================
with gr.Blocks(title="Meeting Minutes AI") as demo:
    gr.Markdown("# 🎙️ Open-Source Meeting Minutes Generator\n**Whisper + Llama-3.2-3B** — Fully Free & Private")

    audio_input = gr.Audio(sources=["upload", "microphone"], type="filepath", label="Upload or Record Audio", format="mp3")

    btn = gr.Button("🚀 Generate Meeting Minutes", variant="primary", size="large")

    with gr.Row():
        with gr.Column(scale=1):
            transcription_output = gr.Textbox(label="📝 Full Transcription", lines=14, show_copy_button=True)
        with gr.Column(scale=2):
            status = gr.Textbox(label="Status", value="Ready", interactive=False)
            minutes_output = gr.Markdown(label="📋 Professional Meeting Minutes",
                show_copy_button=True)

    pdf_download = gr.DownloadButton("📥 Download as PDF", size="large")

    btn.click(
        fn=process_meeting,
        inputs=audio_input,
        outputs=[transcription_output, status, minutes_output, pdf_download]
    )

    gr.Markdown("---\nBuilt with ❤️ using Whisper + Llama-3.2-3B")

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://11be9493ab5a298648.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
